---
title: SQLAlchemy & JupySQL
subtitle: SQL Lab in JupyterLab
# license: CC-BY-4.0
github: https://github.com/magus1968/learning-sql
subject: Technical Portfolio
# subject: SQL Learning & Tooling
venue: GitHub & GitVerse Pages
abstract: |
  Настройка подключения к учебной базе данных Sakila и выполнения SQL-запросов **в блокнотах JupyterLab**.
authors:
  - name: Alex Smirnov
    email: a@smirnovs.pro
    corresponding: true
    affiliations: Data & BI Analyst
date: 2026-11-12
abbreviations:
    MyST: Markedly Structured Text
    ПКМ: Правая Кнопка Мыши
    Jupyter Book: Build static Web-books
    JupySQL: Run & highlight SQL in Jupyter
    Pandas: Библиотека Python для анализа и обработки данных
    Polars: Мощный аналог Pandas на Rust/Python
---

После развертывания сервера MySQL и базы данных Sakila настроим рабочее пространство, чтобы сделать JupyterLab независимой автономной средой для работы с SQL без необходимости прибегать к внешним клиентам  вроде DBeaver, Workbench или DataGrip.

## Создание подключения

Для подключения к базе данных Sakila и работы с ней в JupyterLab, создадим стартовую ячейку инициализации. Она связывает между собой Python, драйвер PyMySQL, библиотеку SQLAlchemy и расширение JupySQL, позволяя выполнять SQL-код прямо в ячейках блокнота:

In [1]:
# 1. Импорт библиотек
import pandas as pd
import sqlalchemy as sa
import sql

# 2. Настройка отображения таблиц в Pandas
# Снижаем порог усечения с дефолтных 60 до 20 строк
pd.set_option('display.max_rows', 20)

# 3. Активация и настройка SQL-магии (JupySQL)
# Загрузка расширения JupySQL в ядро блокнота
%load_ext sql
# Автоконвертация результата запроса в pandas.DataFrame
%config SqlMagic.autopandas = True
# Отключение служебной строки 'Running query in...'
%config SqlMagic.displaycon = False
# Не активен при autopandas=True; лимит (default = 10) передан в Pandas
# %config SqlMagic.displaylimit = 20
# Отключение счетчика затронутых строк 'X rows affected' (по умолчанию включен)
# %config SqlMagic.feedback = False

# 4. Создание подключения через SQLAlchemy
# Сборка объекта URL с параметрами подключения к базе данных
connection_url = sa.engine.URL.create(
    drivername='mysql+pymysql',
    host='localhost',
    port=3306,
    database='sakila',
    username='root',
    password='********',
)
# Создание движка подключения SQLAlchemy Engine на основе сформированного URL
engine = sa.create_engine(connection_url)

# 5. Передача созданного движка подключения (Engine) в JupySQL
%sql engine

# 6. Контроль версий библиотек и статуса подключения
print(f"Pandas ver. {pd.__version__}: порог усечения строк уменьшен до 20")
print(f"SQLAlchemy ver. {sa.__version__}: подключение создано")
print(f"JupySQL ver. {sql.__version__}: подключен через SQLAlchemy Engine")

Pandas ver. 3.0.5: порог усечения строк уменьшен до 20
SQLAlchemy ver. 2.0.52: подключение создано
JupySQL ver. 0.11.1: подключен через SQLAlchemy Engine


### Разбор блока импортов

```python
# 1. Импорт библиотек
import pandas as pd
import sqlalchemy as sa
import sql
```

`import pandas as pd`
- Импортируем библиотеку **Pandas** – инструмент для анализа табличных данных и работы с датафреймами (DataFrame).
- **В контексте книги** используем для усечения многострочного вывода результатов запроса: чтобы вывод, как в книге, отображал через многоточие только первые и последние пять строк вместо вывода всех.

`import sqlalchemy as sa`
- Импортируем библиотеку **SQLAlchemy** – промышленный стандарт Python для управления подключениями и работы с реляционными базами данных.

`import sql`
- Импортируем модуль **sql** библиотеки **JupySQL**[^1].
- В коде используем только ради вывода версии JupySQL (`sql.__version__`).
- Для работы самой SQL-магии (`%load_ext sql` и `%sql engine`) эта строка **не требуется** – расширение загружается ядром Jupyter независимо.

[^1]: Несмотря на то, что библиотека при установке через Conda называется `jupysql`, разработчики библиотеки решили, что внутри Python ее корневой модуль для импорта будет называться коротко – `sql` (так исторически повелось от ее предшественника `ipython-sql`).